In [13]:
import pandas as pd
import numpy as np
import os

# 1. CONFIGURATION

In [14]:
# Key = Table Row Name, Value = Directory Name
DATASETS = {
    "\\textit{linearly separable}": "logLossLS",
    "\\textit{bars \& stripes}": "logLossBS",
    "\\textit{two-curves}": "logLossTC",
    "\\textit{MNIST PCA}": "logLossMN",
    "\\textit{MNIST $28 \\times 28$}": "logLossMN_28x28",
}

BASE_DIR = "."  # Adjust if your folders are elsewhere
MAX_GATES = 125 # How many classical files to check

# 2. DATA LOADING HELPERS

In [15]:
def get_classical_metrics(dataset_dir):
    """
    Scans ACC_classical_surrogate_{n}.txt files.
    Returns best (Train, Test) tuple based on the highest Test accuracy found.
    """
    best_test = -1.0
    best_train = -1.0

    path = os.path.join(BASE_DIR, dataset_dir)
    if not os.path.exists(path): return np.nan, np.nan

    # Iterate through all gate files
    for n in range(1, MAX_GATES + 1):
        file_path = os.path.join(path, f"ACC_classical_surrogate_{n}.txt")
        if not os.path.exists(file_path): continue

        try:
            # Parse file (skipping first line metadata)
            with open(file_path, 'r') as f:
                lines = f.readlines()[1:] # Skip header

            for line in lines:
                if not line.strip(): continue
                parts = line.split()
                # Parse dict-like line: "train_acc 0.9 val_acc 0.85 ..."
                data = dict(zip(parts[::2], parts[1::2]))

                # Get values
                train = float(data.get('train_acc', 0.0))
                test = float(data.get('val_acc', 0.0))

                # Update if any better
                if test > best_test:
                    best_test = test
                if train > best_train:
                    best_train = train
        except Exception:
            continue

    return best_train, best_test

In [16]:
def get_qnn_metrics(dataset_dir):
    """
    Reads ACC_test.txt (Test Acc) and ACC_train.txt (Train Acc) for QNN.
    Assumes both files have the same structure (columns 0 & 1 are class accuracies).
    Returns (Best_Train, Best_Test).
    """
    path_test = os.path.join(BASE_DIR, dataset_dir, "ACC_test.txt")
    path_train = os.path.join(BASE_DIR, dataset_dir, "ACC_train.txt")

    # 1. Load Test Accuracy (to find the BEST epoch)
    try:
        data_test = np.genfromtxt(path_test, skip_header=1)
        if data_test.ndim == 1: data_test = data_test.reshape(1, -1)

        # Calculate Average Test Acc per epoch
        avg_test = (data_test[:, 0] + data_test[:, 1]) * 0.5
        best_idx = np.argmax(avg_test) # Index of the best test epoch
        best_test_val = avg_test[best_idx]

    except Exception:
        return np.nan, np.nan

    # 2. Load Train Accuracy
    try:
        data_train = np.genfromtxt(path_train, skip_header=1)
        if data_train.ndim == 1: data_train = data_train.reshape(1, -1)

        # Calculate Average Train Acc
        avg_train = (data_train[:, 0] + data_train[:, 1]) * 0.5
        best_idx = np.argmax(avg_train) # Index of the best train epoch
        best_train_val = avg_train[best_idx]

    except Exception:
        best_train_val = np.nan

    return best_train_val, best_test_val

# 3. GENERATE TABLE

In [17]:
rows = []

for name, directory in DATASETS.items():
    # 1. Get metrics
    c_train, c_test = get_classical_metrics(directory)
    q_train, q_test = get_qnn_metrics(directory)

    # 2. Format numbers as Percentages (multiply by 100, add \%)
    # Classical
    c_train_s = f"{c_train * 100:.1f}\%"
    c_test_s = f"{c_test * 100:.1f}\%"

    # QNN
    q_train_s = f"{q_train * 100:.1f}\%" if not np.isnan(q_train) else "N/A"
    q_test_s = f"{q_test * 100:.1f}\%"

    # 3. BOLDING LOGIC (Compare Test Accuracies)
    if c_test > q_test:
        c_test_s = f"\\textbf{{{c_test_s}}}"  # Apply bold wrapper
    elif q_test > c_test:
        q_test_s = f"\\textbf{{{q_test_s}}}"

    rows.append([c_train_s, c_test_s, q_train_s, q_test_s])

# 4. Create DataFrame with MultiIndex Columns
columns = pd.MultiIndex.from_tuples([
    ('Classical Surrogate', 'Train'),
    ('Classical Surrogate', 'Test'),
    ('QNN', 'Train'),
    ('QNN', 'Test')
])

df = pd.DataFrame(rows, index=DATASETS.keys(), columns=columns)
df

Classical Surrogate                      QNN  \
                                            Train             Test   Train   
\textit{linearly separable}                90.3\%           84.4\%  98.1\%   
\textit{bars \& stripes}                   95.0\%           92.1\%  94.0\%   
\textit{two-curves}                        98.3\%  \textbf{98.4\%}  87.6\%   
\textit{MNIST PCA}                         94.0\%  \textbf{94.5\%}  92.5\%   
\textit{MNIST $28 \times 28$}              99.6\%           91.6\%  98.4\%   

                                                
                                          Test  
\textit{linearly separable}    \textbf{97.3\%}  
\textit{bars \& stripes}       \textbf{96.3\%}  
\textit{two-curves}                     87.6\%  
\textit{MNIST PCA}                      93.6\%  
\textit{MNIST $28 \times 28$}  \textbf{92.9\%}

In [18]:
# 5. Export to LaTeX
latex_code = df.to_latex(
    escape=False, # Essential for \textbf{}
    column_format="lcccc",
    caption="Comparison of training and test accuracies in classical surrogate benchmark of Q-FLAIR QNNs.",
    label="tab:classical_surrogate_bench",
    multicolumn_format="c",
)

print(latex_code)

\begin{table}
\caption{Comparison of training and test accuracies in classical surrogate benchmark of Q-FLAIR QNNs.}
\label{tab:classical_surrogate_bench}
\begin{tabular}{lcccc}
\toprule
 & \multicolumn{2}{c}{Classical Surrogate} & \multicolumn{2}{c}{QNN} \\
 & Train & Test & Train & Test \\
\midrule
\textit{linearly separable} & 90.3\% & 84.4\% & 98.1\% & \textbf{97.3\%} \\
\textit{bars \& stripes} & 95.0\% & 92.1\% & 94.0\% & \textbf{96.3\%} \\
\textit{two-curves} & 98.3\% & \textbf{98.4\%} & 87.6\% & 87.6\% \\
\textit{MNIST PCA} & 94.0\% & \textbf{94.5\%} & 92.5\% & 93.6\% \\
\textit{MNIST $28 \times 28$} & 99.6\% & 91.6\% & 98.4\% & \textbf{92.9\%} \\
\bottomrule
\end{tabular}
\end{table}



In [20]:
# Print the training vs test accuracy differences per dataset and per classical surrogate vs QNN:
for idx, row in df.iterrows():
    c_train = float(row[('Classical Surrogate', 'Train')].replace('\\%', '').replace('\\textbf{', '').replace('}', ''))
    c_test = float(row[('Classical Surrogate', 'Test')].replace('\\%', '').replace('\\textbf{', '').replace('}', ''))
    q_train = float(row[('QNN', 'Train')].replace('\\%', '').replace('\\textbf{', '').replace('}', '')) if row[('QNN', 'Train')] != "N/A" else np.nan
    q_test = float(row[('QNN', 'Test')].replace('\\%', '').replace('\\textbf{', '').replace('}', ''))

    print(f"{idx}: Classical Train-Test Diff: {c_train - c_test:.1f}%, QNN Train-Test Diff: {q_train - q_test:.1f}%")

\textit{linearly separable}: Classical Train-Test Diff: 5.9%, QNN Train-Test Diff: 0.8%
\textit{bars \& stripes}: Classical Train-Test Diff: 2.9%, QNN Train-Test Diff: -2.3%
\textit{two-curves}: Classical Train-Test Diff: -0.1%, QNN Train-Test Diff: 0.0%
\textit{MNIST PCA}: Classical Train-Test Diff: -0.5%, QNN Train-Test Diff: -1.1%
\textit{MNIST $28 \times 28$}: Classical Train-Test Diff: 8.0%, QNN Train-Test Diff: 5.5%
